In [ ]:
#installing libraries
import pandas as pd;
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# data importation

data = pd.read_csv("Dataset.csv");
print(data.head(5))

In [ ]:
print(data.columns.tolist)
print("******************************")
print(data.describe)

In [ ]:
#missing value computation
print(data.isnull().sum())
print(data.columns.tolist())

In [ ]:
#combinin the relevant features for content-based recommmendations

In [ ]:
def column_features(row):
    return f"{row['type']} {row['title']} {row['country']} {row['director']} {row['listed_in']}"
data['column_features'] = data.apply(column_features, axis=1)
print(data['column_features'].tolist())



In [ ]:
#converting text data into machine readable
#initalize the vectrorizer
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(data['column_features'])
print(tfidf_matrix)

In [ ]:
#comouted similarities

similarity_score = cosine_similarity(tfidf_matrix,tfidf_matrix)
print(similarity_score)

In [ ]:
# recommendations

def get_recommendations(title, data, similarity_score, top_n=5):
    # 1. Check if title exists
    if title not in data['title'].values:
        return f"title {title} is not in the dataset, no recommendations can be generated."

    # 2. Get the integer index o
    index = data[data['title'] == title].index[0]

    # 3. Extract the row from your similarity matrix for this specific index 
    sim_score = list(enumerate(similarity_score[index]))
    
    # 4. Sort by the similarity values 
    sim_score = sorted(sim_score, key=lambda x: x[1], reverse=True)
    
    # 5. Get top matches 
    top_matches = sim_score[1:top_n + 1]

    # 6. Extract target indices and map back to your dataframe 
    movie_indices = [i[0] for i in top_matches]
    return data['title'].iloc[movie_indices].tolist()

# Test run
print(get_recommendations("The wolf", data, similarity_score))




In [ ]:
print(data.columns.tolist())
print(data['country'].isnull().sum())
print(data['type'].isnull().sum())


In [ ]:
#task 2 content type prediction model

x = data[['release_year','date_added','listed_in' ]]
print("relearse year:", data['release_year'].values)
y= data['type']


#encoding feature for (y)  and target (y)

from sklearn.preprocessing import LabelEncoder
from sklearn import preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.model_selection import train_test_split



#using lableEncoder for catergory variables

le = preprocessing.LabelEncoder();


y_encoded = le.fit_transform(y)

print(y_encoded)

print("categorical varaibles encoded")

X_train,X_test,Y_train,y_test = train_test_split(x,y_encoded,test_size=0.2,random_state=42)


preprocessor = ColumnTransformer(
    transformers= [
        ('num', StandardScaler(),['release_year']),
        ('cat', OneHotEncoder(handle_unknown='ignore',sparse_output=False),['listed_in',])
    ]
)


X_train_processed= preprocessor.fit_transform(X_train)
X_test_processed= preprocessor.transform(X_test)


# 1. Print the raw value of the first item in your original data
print("--- BEFORE SCALING (Original Data) ---")
print(X_train['release_year'].iloc[0])

# 2. Print the first column of the processed matrix which holds 'release_year'
print("\n--- AFTER SCALING (Processed Matrix Row 0, Column 0) ---")
print(X_train_processed[0, 0])



In [ ]:
#model designing and evaluation


from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

#LogisticRegression model implementation
logistic_model = LogisticRegression(random_state=42)
logistic_model.fit(X_train_processed, Y_train)

#decisionTree model
decision_tree_model=  DecisionTreeClassifier(random_state=42)
decision_tree_model.fit(X_train_processed,Y_train)


In [ ]:
#Evaluating the model accuracy, and prediction power


from sklearn.metrics import accuracy_score, classification_report

logistic_model_prediction = logistic_model.predict(X_test_processed)
decision_tree_prediction= decision_tree_model.predict(X_test_processed)




print("=" * 50)
print(f"LOGISTIC REGRESSION ACCURACY: :{accuracy_score(y_test,logistic_model_prediction):.2%}")
print(classification_report(y_test,logistic_model_prediction,target_names=['Movie', 'TV Show']))
print("*" * 50)


print("=" * 50)
print(f"Decision Tree Accuracy: :{accuracy_score(y_test,decision_tree_prediction):.2%}")
print(classification_report(y_test,decision_tree_prediction,target_names=['Movie', 'TV Show']))
print("*" * 50 )

In [ ]:
#feature importance

categorical_encoder = preprocessor.named_transformers_['cat']
print("feature_in", categorical_encoder.feature_names_in_)
encoded_cat_names = categorical_encoder.get_feature_names_out(['listed_in'])
all_feature_names = ['release_year'] + list(encoded_cat_names)

#match with the importance

# match with the importance
importance = decision_tree_model.feature_importances_

importance_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Importance': importance      # <-- Created with a CAPITAL 'I'
})

# FIXED: Changed 'importance' to 'Importance' to match the case exactly
importance_df = importance_df.sort_values(by='Importance', ascending=False)

print("=" * 40)
print("TOP 5 MOST IMPORTANT COLUMNS:")
print("=" * 40)
print(importance_df.head(5))


